In [2]:
pip install pandas cryptography openpyxl


Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd
import random

# Generate file1.xlsx
data1 = {"Values": [random.randint(1, 100) for _ in range(500)]}
df1 = pd.DataFrame(data1)
df1.to_excel("file1.xlsx", index=False)

# Generate file2.xlsx
data2 = {"Values": [random.randint(1, 100) for _ in range(500)]}
df2 = pd.DataFrame(data2)
df2.to_excel("file2.xlsx", index=False)

print("Excel files generated successfully!")


Excel files generated successfully!


In [7]:
from cryptography.fernet import Fernet

# Generate a valid Fernet key
key = Fernet.generate_key()
print(f"Generated Fernet Key: {key.decode()}")

# Save the key for consistent use
with open("fernet_key.key", "wb") as key_file:
    key_file.write(key)



Generated Fernet Key: cU7F7a-gGeFwRrltxq3WB8JV3WMEBR9uBELN3ZNpULU=


In [13]:
import socket
import threading
import pandas as pd
from cryptography.fernet import Fernet

# Load Fernet key
with open("fernet_key.key", "rb") as key_file:
    key = key_file.read()

cipher = Fernet(key)

def start_server(port, file_name):
    server = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    server.bind(('localhost', port))
    server.listen(1)
    print(f"Server hosting {file_name} listening on port {port}...")

    while True:
        conn, addr = server.accept()
        print(f"Connection from {addr}")

        # Receive file request
        requested_file = conn.recv(1024).decode()
        if requested_file == file_name:
            # Read Excel file and compute sum
            data = pd.read_excel(file_name)
            total_sum = data.iloc[:, 0].sum()

            # Encrypt the sum
            encrypted_sum = cipher.encrypt(str(total_sum).encode())
            conn.send(encrypted_sum)
        else:
            conn.send(b"Invalid file request")
        conn.close()


In [15]:
def request_sum(server_ip, port, file_name):
    client = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    client.connect((server_ip, port))
    client.send(file_name.encode())

    # Receive encrypted sum
    encrypted_sum = client.recv(1024)
    client.close()

    # Decrypt and return the sum
    decrypted_sum = cipher.decrypt(encrypted_sum).decode()
    return float(decrypted_sum)

def client_task():
    # Request sums from both servers
    sum1 = request_sum('localhost', 5000, 'file1.xlsx')
    sum2 = request_sum('localhost', 6000, 'file2.xlsx')

    # Compute final sum
    final_sum = sum1 + sum2
    print(f"Final sum from both servers: {final_sum}")


In [17]:
# Start Server 1
server1_thread = threading.Thread(target=start_server, args=(5000, 'file1.xlsx'), daemon=True)
server1_thread.start()

# Start Server 2
server2_thread = threading.Thread(target=start_server, args=(6000, 'file2.xlsx'), daemon=True)
server2_thread.start()

# Start Client
client_thread = threading.Thread(target=client_task)
client_thread.start()

# Wait for the client to finish
client_thread.join()


Server hosting file1.xlsx listening on port 5000...
Server hosting file2.xlsx listening on port 6000...
Connection from ('127.0.0.1', 50114)
Connection from ('127.0.0.1', 50115)
Final sum from both servers: 50733.0
